# 第十二章：AI Agent 系统

## 从回答问题到自主执行任务

LLM 的下一阶段进化是**AI Agent**——不仅能理解和生成文本，还能**使用工具、制定计划、自主完成任务**。

```
传统 LLM:   输入文本 → 输出文本
AI Agent:   目标描述 → 规划 → 调用工具 → 观察结果 → 迭代 → 完成
```

## 12.1 Agent 核心概念

### ReAct 模式 (Reasoning + Acting)

Yao et al. (2022) 提出 ReAct——将**推理**和**行动**交织：

```
Thought: 我需要知道今天北京的温度
Action:  weather_api("北京", "今天")
Observation: 北京今天 28°C，晴
Thought: 已经获得温度数据，可以回答
Answer: 北京今天 28°C，晴天
```

ReAct 的关键洞见：**推理帮助选择正确的行动，行动结果反过来指导推理**。

### Agent 的记忆系统

| 记忆类型 | 存储内容 | 实现方式 |
|----------|---------|---------|
| **短期记忆** | 当前对话上下文 | 直接拼入 prompt |
| **工作记忆** | 中间推理和行动结果 | scratchpad，ReAct 的 Thought/Action/Observation |
| **长期记忆** | 用户偏好、历史会话 | 向量数据库 + 摘要 |

### Plan-and-Execute

复杂任务直接一步到位容易出错。**先规划后执行**：

```
目标: 分析 Q2 销售数据并写报告

Plan:
  1. 查询 Q2 销售数据库 → 获取原始数据
  2. 计算同比/环比增长率
  3. 识别 top-5 增长和下降产品
  4. 生成数据可视化图表
  5. 撰写分析报告

Execute: 逐步执行计划，每步验证结果
```

In [ ]:
# === ReAct Agent 核心循环（伪代码框架） ===
class ReActAgent:
    def __init__(self, llm, tools):
        self.llm = llm          # LLM 调用函数
        self.tools = tools      # {"tool_name": tool_function}
        self.max_steps = 10
    
    def run(self, task):
        history = [f"任务: {task}"]
        
        for step in range(self.max_steps):
            # 1. LLM 决定下一步
            prompt = self._build_prompt(history)
            response = self.llm(prompt)
            
            # 2. 解析动作
            action = self._parse_action(response)
            
            if action["type"] == "answer":
                return action["content"]
            
            # 3. 执行工具调用
            tool_result = self.tools[action["tool"]](action["input"])
            
            # 4. 记录结果，进入下一轮
            history.append(f"Observation: {tool_result}")
        
        return "达到最大步数限制"

# 典型 prompt 模板:
REACT_PROMPT = """你可以使用以下工具:
{tools}

使用以下格式:
Thought: 当前需要做什么
Action: 工具名称
Action Input: 工具输入
Observation: 工具返回结果
... (可重复)
Thought: 我现在知道答案了
Final Answer: 最终回答

任务: {task}
{history}"""


## 12.2 Tool Use / Function Calling

### 函数调用协议

现代 LLM API 原生支持 **Function Calling**——模型输出结构化的函数调用请求，而非自然语言：

```json
{
  "tool_calls": [{
    "function": {
      "name": "get_weather",
      "arguments": "{\"city\": \"北京\", \"date\": \"2026-07-16\"}"
    }
  }]
}
```

### 工具设计原则

1. **单一职责**：每个工具只做一件事
2. **明确的输入输出 schema**：用 JSON Schema 描述参数类型和约束
3. **幂等性**：只读工具 (GET) 优先，写操作 (POST) 需人工确认
4. **错误可恢复**：返回结构化错误信息，Agent 可据此重试

| 工具类别 | 示例 |
|---------|------|
| **检索** | 搜索文档、查询数据库、调用 API |
| **计算** | 执行代码、数学计算、数据分析 |
| **操作** | 发送邮件、创建文件、调用外部服务 |
| **UI** | 点击按钮、填写表单、截图 |

## 12.3 多 Agent 系统

### 为什么需要多 Agent？

单个 Agent 受限于上下文窗口和工具集。多个 Agent 分工协作：

| 架构 | 模式 | 适用场景 |
|------|------|---------|
| **层级式** | 管理者 Agent 分配任务给子 Agent | 复杂任务分解 |
| **流水线** | Agent A 输出 → Agent B 输入 | 数据处理链路 |
| **辩论式** | 多个 Agent 从不同角度分析，投票决策 | 需要多视角的任务 |
| ** swarm** | 动态组建 Agent 团队 | 开放式探索 |

### LangGraph：有状态的 Agent 编排

LangGraph 用**有向图**定义 Agent 流程，节点是 Agent/工具，边是条件跳转：

```
        ┌──────────┐
        │  START   │
        └────┬─────┘
             ↓
        ┌──────────┐    需要工具?    ┌──────────┐
        │  Agent   │──────────────→  │  Tools   │
        └────┬─────┘                 └────┬─────┘
             │ 完成                       │
             ↓                            ↓
        ┌──────────┐              ┌──────────┐
        │   END    │              │  Agent   │ (循环)
        └──────────┘              └──────────┘
```

与传统链式调用 (LangChain) 的区别：LangGraph 支持**循环、条件分支、状态持久化**——真正适合 Agent 的不确定执行路径。

## 12.4 Agent 评估

| 指标 | 含义 | 测量方式 |
|------|------|---------|
| **任务完成率** | 是否达成用户目标 | 人工评估或 LLM-as-judge |
| **工具调用准确率** | 选择的工具和参数是否正确 | 对比 ground truth |
| **执行效率** | 完成任务所需的步骤数和 token 消耗 | 日志统计 |
| **安全性** | 是否执行了危险操作 | 沙箱测试 + 规则引擎 |

## 本章小结

| 概念 | 关键点 |
|------|--------|
| **ReAct** | 推理与行动交织，Thought→Action→Observation 循环 |
| **Tool Use** | 函数调用协议，结构化的工具接口 |
| **多 Agent** | 层级/流水线/辩论式架构，分工协作 |
| **LangGraph** | 有向图编排，支持循环和状态持久化 |

> **下一章**：第十三章——多模态 AI，将文本模型扩展到图像、语音等多种模态。